In [2]:
# Add the root of the repository to the Python path
repo_root = os.path.abspath('../../')
if repo_root not in sys.path:
    sys.path.append(repo_root)

In [3]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import Normalizer
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from src.part2_pipeline.pipeline import build_feature_pipeline

In [ ]:
# Init local spark
spark = SparkSession.builder.appName("Task3_SVM_Dev").master("local[*]").getOrCreate()

# Build the feature pipeline
base_pipeline = build_feature_pipeline(num_top_features=2000)

# Create the normalizer
normalizer = Normalizer(inputCol="selected_features", outputCol="normalizedFeatures", p=2.0)

# Create SVM
svm = LinearSVC(featuresCol="normalizedFeatures", labelCol="label", maxIter=10, regParam=0.1)

#Wrap in OneVsRest for Multi-Class support
ovr = OneVsRest(classifier=svm, featuresCol="normalizedFeatures", labelCol="label")

# Assemble the final pipeline
final_stages = base_pipeline.getStages() + [normalizer, ovr]
final_pipeline = Pipeline(stages=final_stages)